In [1]:
import pandas as pd

In [2]:
# Load the two datasets

CRIME_DIR = "./Data/crime_data"
COOK_DIR = "./Data/Cook_county_data"

crime_df = pd.read_csv(f"{CRIME_DIR}/Homicides_Since_2020.csv")
cook_df = pd.read_parquet(f"{COOK_DIR}/chicago_sales_modeling.parquet")

cook_df.columns = cook_df.columns.str.lower().str.strip()
crime_df.columns = crime_df.columns.str.lower().str.strip()

In [3]:
# Rename columns to 'community_area' in both dataframes

cook_df = cook_df.rename(columns={'chicago_community_area_num': 'community_area'})
crime_df = crime_df.rename(columns={'community area': 'community_area'})

In [4]:
# Cast each column to int

crime_df = crime_df.dropna(subset=['community_area'])
crime_df['community_area'] = crime_df['community_area'].astype(int)

cook_df = cook_df.dropna(subset=['community_area'])
cook_df['community_area'] = cook_df['community_area'].astype(int)

In [5]:
# Create a sale_year column in cook_df

cook_df['sale_date_parsed'] = pd.to_datetime(cook_df['sale_date'], format='%B %d, %Y', errors='coerce')
cook_df['sale_year'] = cook_df['sale_date_parsed'].dt.year
cook_df = cook_df.dropna(subset=['sale_year'])
cook_df['sale_year'] = cook_df['sale_year'].astype(int)

We will create two crime-related columns for the merged dataset:

(1) The first column associates with each property sale the total amount of homicides in that property's community area since 2020.

(2) Since that is somewhat anachronistic (or rather, it pretends to compute a time-independent "homicide score" for each community area), we also associate to each property sale a column that counts the total amount of homicides in that property's community area in the year prior to the sale.

In [6]:
# Implementing (1)

total_crime = (
    crime_df.groupby('community_area')
    .size()
    .reset_index(name='total_homicides_since_2020')
)

merged_df = pd.merge(cook_df, total_crime, on='community_area', how='left')

In [7]:
# Implementing (2)

# TO DO

In [8]:
# Save the merged dataframe to a parquet file
merged_df.to_parquet("./Data/sale_and_crime_data/chicago_sales_with_crime.parquet", index=False)